##Run config first

In [0]:
%run "../00_config"

In [0]:
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F

##fetch function

In [0]:

ALL_json_products = []
files_info = dbutils.fs.ls(PRODUCTS_VOLUME)

for file in files_info:
    if file.name.startswith("products_"):
        with open(f"{PRODUCTS_VOLUME}/{file.name}", "r") as f:
            products = json.load(f)
            for product in products:
                ALL_json_products.append({
                    "category": file.name.split("_", 1)[1].split(".")[0],
                    **product
                })

print(ALL_json_products)

## fetch products

In [0]:
all_products = []
 
for p in ALL_json_products:
    all_products.append({
        "asin":               p.get("asin", ""),
        "product_title":      p.get("title", ""),
        "position":           int(p.get("position") or 0),
        "current_price_sar":  float(p.get("price", {}).get("currentPrice") or 0.0),
        "before_price_sar":   float(p.get("price", {}).get("beforePrice") or 0.0),
        "currency":           p.get("price", {}).get("symbol", "SAR"),
        "rating":             float(p.get("reviews", {}).get("rating") or 0.0),
        "total_reviews":      int(p.get("reviews", {}).get("totalReviews") or 0),
        "is_amazon_choice":   bool(p.get("badges", {}).get("amazonChoice", False)),
        "is_best_seller":     bool(p.get("badges", {}).get("bestSeller", False)),
        "is_prime":           bool(p.get("badges", {}).get("amazonPrime", False)),
        "is_sponsored":       p.get("isSponsored", ""),
        "bought_past_month":  p.get("boughtInPastMonth", ""),
        "stock_warning":      p.get("deliveryInfo", {}).get("stockWarning", ""),
        "product_photo":      p.get("image", ""),
        "product_url":        p.get("url", ""),
        "category":           p.get("category", ""),
        "_snapshot_date":     SNAPSHOT_DATE,
        "_ingested_at":       datetime.now(timezone.utc).strftime("%d-%m-%Y %H:%M:%S")
    })
 
print(f"\nTotal products collected: {len(all_products)}")

##Write to Bronze

In [0]:
df_bronze = spark.createDataFrame(all_products)
display(df_bronze)

(df_bronze.write
    .format("delta")
    .mode("Overwrite")
    .saveAsTable(BRONZE_PRODUCTS)
)

print(f"✅ {df_bronze.count()} products written to {BRONZE_PRODUCTS}")

##Validate

In [0]:
%sql
SELECT *
FROM saudi_retail_catalog.bronze.bronze_product
WHERE asin in ( "B0G6KKV813",
"B0F8R17HC9",
"B09C66BB2G",
"B0FT31CT6B",
"B0CDBPTF4L",
"B0FZV7STQ6",
"B08KXMG6ZY",
"B0GVGNSHK8",
"B0D6399BJL",
"B0G6KW6J9S",
"B0GS533BBL")
ORDER by asin
    -- SELECT 
    -- asin, count(*)
    -- FROM saudi_retail_catalog.bronze.bronze_product
    -- GROUP BY asin
    -- HAVING count(*) > 1
    -- ORDER BY count(*) DESC
-- )
-- WHERE current_price_sar = 0 OR asin = "" OR current_price_sar IS NULL;